# **NLP Practical 3 — Evaluation of Named Entity Recognition (NER) Model using Confusion Matrix**

---

**Name: Shubham Sarwar**  
**PRN: 202301040127**

**Problem Statement:** Train an NER model using an NLP library (spaCy / NLTK) and
evaluate its performance using a **Confusion Matrix**, computing Precision, Recall
and F1-score for every entity type.

### Learning Objectives
- Understand the concept of Named Entity Recognition (NER).
- Use a pre-trained NER model on a custom test set of 20 sentences.
- Compute and interpret a confusion matrix for NER.
- Calculate Precision, Recall and F1-score.
- Perform error analysis and suggest improvements.


## **1. Install spaCy and the English Model**

In [ ]:
!pip install -q spacy scikit-learn pandas seaborn matplotlib
!python -m spacy download en_core_web_sm

## **2. Import Libraries and Load the Model**

In [ ]:
import spacy
from spacy import displacy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_recall_fscore_support,
    accuracy_score,
)

# Load the pre-trained English NER pipeline
nlp = spacy.load("en_core_web_sm")
print("spaCy model loaded:", nlp.meta["name"])

## **3. Quick Demo — How NER Works**

Common entity labels spaCy uses:

| Label   | Meaning                                       |
|---------|-----------------------------------------------|
| PERSON  | People (real or fictional)                    |
| ORG     | Companies, agencies, institutions             |
| GPE     | Countries, cities, states (geo-political)     |
| LOC     | Non-GPE locations (mountains, lakes)          |
| DATE    | Dates                                         |
| MONEY   | Monetary values                               |


In [ ]:
sample = "Sundar Pichai is the CEO of Google located in California."
doc = nlp(sample)

print("Sample NER output:")
for ent in doc.ents:
    print(f"  {ent.text:20s} -> {ent.label_:8s} ({spacy.explain(ent.label_)})")

# Render an inline visual with colored entity highlighting
displacy.render(doc, style="ent", jupyter=True)

## **4. Build the Annotated Test Dataset (20 sentences)**

We manually label the true entities in each sentence. The list-of-tuples format
`(entity_text, true_label)` is simple and easy to grade against the model's predictions.

> Note: only the **most important entities** in each sentence are annotated, which is
> standard practice. Common labels used: `PERSON, ORG, GPE, DATE, MONEY, LOC, PRODUCT`.


In [ ]:
# 20 test sentences with their ground-truth entities
test_data = [
    ("Sundar Pichai is the CEO of Google in California.",
     [("Sundar Pichai", "PERSON"), ("Google", "ORG"), ("California", "GPE")]),

    ("Narendra Modi visited New Delhi on Monday.",
     [("Narendra Modi", "PERSON"), ("New Delhi", "GPE"), ("Monday", "DATE")]),

    ("Apple announced a new iPhone in Cupertino last September.",
     [("Apple", "ORG"), ("Cupertino", "GPE"), ("last September", "DATE")]),

    ("Elon Musk founded SpaceX in 2002 in Hawthorne.",
     [("Elon Musk", "PERSON"), ("SpaceX", "ORG"), ("2002", "DATE"), ("Hawthorne", "GPE")]),

    ("Microsoft acquired GitHub for $7.5 billion in 2018.",
     [("Microsoft", "ORG"), ("GitHub", "ORG"), ("$7.5 billion", "MONEY"), ("2018", "DATE")]),

    ("Barack Obama was born in Hawaii in August 1961.",
     [("Barack Obama", "PERSON"), ("Hawaii", "GPE"), ("August 1961", "DATE")]),

    ("Amazon opened a new warehouse in Mumbai last week.",
     [("Amazon", "ORG"), ("Mumbai", "GPE"), ("last week", "DATE")]),

    ("Mark Zuckerberg leads Meta from Menlo Park.",
     [("Mark Zuckerberg", "PERSON"), ("Meta", "ORG"), ("Menlo Park", "GPE")]),

    ("Tesla shares rose 5 percent on Friday after the Berlin announcement.",
     [("Tesla", "ORG"), ("5 percent", "PERCENT"), ("Friday", "DATE"), ("Berlin", "GPE")]),

    ("ISRO launched Chandrayaan-3 from Sriharikota in July 2023.",
     [("ISRO", "ORG"), ("Chandrayaan-3", "PRODUCT"), ("Sriharikota", "GPE"), ("July 2023", "DATE")]),

    ("J.K. Rowling wrote Harry Potter in Edinburgh.",
     [("J.K. Rowling", "PERSON"), ("Harry Potter", "WORK_OF_ART"), ("Edinburgh", "GPE")]),

    ("Infosys reported revenues of $4 billion last quarter.",
     [("Infosys", "ORG"), ("$4 billion", "MONEY"), ("last quarter", "DATE")]),

    ("The match between India and Australia was held in Melbourne.",
     [("India", "GPE"), ("Australia", "GPE"), ("Melbourne", "GPE")]),

    ("Sam Altman runs OpenAI which is based in San Francisco.",
     [("Sam Altman", "PERSON"), ("OpenAI", "ORG"), ("San Francisco", "GPE")]),

    ("Bill Gates founded Microsoft in 1975 in Albuquerque.",
     [("Bill Gates", "PERSON"), ("Microsoft", "ORG"), ("1975", "DATE"), ("Albuquerque", "GPE")]),

    ("Reliance Industries invested $10 billion in green energy projects.",
     [("Reliance Industries", "ORG"), ("$10 billion", "MONEY")]),

    ("The Eiffel Tower in Paris was built in 1889.",
     [("Eiffel Tower", "FAC"), ("Paris", "GPE"), ("1889", "DATE")]),

    ("Sachin Tendulkar scored a century against England in 2002.",
     [("Sachin Tendulkar", "PERSON"), ("England", "GPE"), ("2002", "DATE")]),

    ("Google DeepMind released a paper about Gemini in December.",
     [("Google DeepMind", "ORG"), ("Gemini", "PRODUCT"), ("December", "DATE")]),

    ("Sundar visited Bengaluru with Sundar Pichai and Satya Nadella.",
     [("Sundar", "PERSON"), ("Bengaluru", "GPE"),
      ("Sundar Pichai", "PERSON"), ("Satya Nadella", "PERSON")]),
]

print(f"Number of test sentences: {len(test_data)}")
print(f"Total ground-truth entities: {sum(len(t[1]) for t in test_data)}")

## **5. Run the Model and Compare with Ground Truth**

For every sentence we collect the model's predicted entities and compare them with
our gold-standard labels. We align by the entity text:

- If both predict and truth contain an entity with the same text → compare labels.
- If truth has the entity but the model missed it → predicted = `O` (a special "no-entity" marker = False Negative).
- If the model predicted an entity that wasn't in truth → truth = `O` (False Positive).

This entity-level comparison is the standard way to evaluate NER.


In [ ]:
y_true, y_pred = [], []
per_sentence_results = []

for sentence, true_entities in test_data:
    doc = nlp(sentence)

    # Build dictionaries: {entity_text: label}
    true_map = {ent_text: lbl for ent_text, lbl in true_entities}
    pred_map = {ent.text: ent.label_ for ent in doc.ents}

    # Union of all entity texts that appear in either
    all_entities = set(true_map.keys()) | set(pred_map.keys())

    sent_rows = []
    for ent in all_entities:
        true_label = true_map.get(ent, "O")     # O = not annotated / missed
        pred_label = pred_map.get(ent, "O")
        y_true.append(true_label)
        y_pred.append(pred_label)
        match = "✓" if true_label == pred_label else "✗"
        sent_rows.append((ent, true_label, pred_label, match))

    per_sentence_results.append((sentence, sent_rows))

# Show first few sentences and how the model did
for sentence, rows in per_sentence_results[:5]:
    print(f"\nSentence: {sentence}")
    print(f"  {'ENTITY':25s} {'TRUE':12s} {'PRED':12s} MATCH")
    for ent, t, p, m in rows:
        print(f"  {ent[:25]:25s} {t:12s} {p:12s}  {m}")

## **6. Build the Confusion Matrix**

In [ ]:
# Collect all labels seen anywhere (in truth or prediction)
labels = sorted(set(y_true) | set(y_pred))
print("All labels:", labels)

# Confusion matrix:
#   rows    = TRUE labels
#   columns = PREDICTED labels
#   value at [i][j] = number of cases where truth=i and prediction=j
cm = confusion_matrix(y_true, y_pred, labels=labels)

# Pretty-print as a pandas DataFrame
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
print("\nConfusion Matrix:")
print(cm_df)

## **7. Visualize the Confusion Matrix (Heatmap)**

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues",
            cbar=True, linewidths=0.5, linecolor="grey")
plt.title("NER Confusion Matrix (rows = Actual, columns = Predicted)")
plt.xlabel("Predicted Label")
plt.ylabel("Actual (True) Label")
plt.tight_layout()
plt.show()

## **8. Calculate Precision, Recall and F1-score**

For each entity type:

$$\text{Precision} = \frac{TP}{TP+FP}, \quad
\text{Recall} = \frac{TP}{TP+FN}, \quad
F_1 = \frac{2 \cdot P \cdot R}{P+R}$$

- **TP (True Positive)** — entity correctly identified with the right label.  
- **FP (False Positive)** — model predicted an entity that wasn't there (or wrong label).  
- **FN (False Negative)** — model missed an entity that was there.


In [ ]:
print("Classification Report (per entity type):\n")
print(classification_report(y_true, y_pred, labels=labels, zero_division=0))

print(f"\nOverall Accuracy: {accuracy_score(y_true, y_pred)*100:.2f}%")

## **9. Per-Entity Metric Table (Custom Format)**

In [ ]:
precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, labels=labels, zero_division=0
)

metrics_df = pd.DataFrame({
    "Label": labels,
    "Precision": np.round(precision, 3),
    "Recall":    np.round(recall, 3),
    "F1-score":  np.round(f1, 3),
    "Support":   support
})
metrics_df = metrics_df.sort_values("Support", ascending=False).reset_index(drop=True)
print(metrics_df.to_string(index=False))

## **10. Error Analysis — Find Where the Model Goes Wrong**

We list every prediction that did not match the ground truth. This is the data we
use to write our error-analysis section of the report.


In [ ]:
# Re-walk through sentences and pull out only the mismatches
errors = []
for sentence, rows in per_sentence_results:
    for ent, t, p, m in rows:
        if m == "✗":
            errors.append({
                "Sentence": sentence,
                "Entity":   ent,
                "True":     t,
                "Predicted": p,
                "Error type": ("missed (FN)" if p == "O"
                               else "extra (FP)" if t == "O"
                               else "wrong label")
            })

err_df = pd.DataFrame(errors)
print(f"Number of errors: {len(err_df)}\n")
print(err_df.to_string(index=False))

### Common error patterns we usually see

1. **Missed entities (FN)** — the model never tagged a real entity. Often happens
   with rare names, made-up products, or non-Western names.
2. **Wrong label** — the model found the entity but put it in the wrong category
   (e.g. tagged a product as ORG, or a city as PERSON).
3. **Extra entities (FP)** — the model invented an entity from a common noun.
4. **Span errors** — the model captured only part of a multi-word entity
   (e.g. "Sundar" instead of "Sundar Pichai"). With entity-text alignment these
   show up as one missed and one extra.

### Ways to improve the model

- **Fine-tune** the model on domain-specific labelled data (medical, legal, Indian-name corpora).
- **Use a bigger model** like `en_core_web_md` or `en_core_web_trf` (transformer-based, much higher accuracy).
- **Add gazetteers / rules** for known lists of names, products, cities (rule-based fallback).
- **Augment training data** with more examples of the entity types that scored low.
- **Handle ambiguity** with context (e.g., "Apple" → ORG when near "iPhone", FOOD otherwise).
- **Post-processing rules** to fix span boundaries (e.g., merge titles like "Dr." with the name).


## **11. Conclusion**

We trained (loaded) a pre-trained spaCy NER model and evaluated it on 20 hand-annotated
sentences. The confusion matrix and per-class metrics reveal:

- The model performs **very well on common categories** (PERSON, ORG, GPE, DATE),
  which dominate the training data of `en_core_web_sm`.
- It is **weaker on rarer / domain-specific categories** like PRODUCT (e.g. Chandrayaan-3,
  Gemini) and WORK_OF_ART (Harry Potter), which are sometimes mis-tagged as ORG.
- Indian and other non-English names sometimes have span issues
  (e.g. "Sundar Pichai" vs "Sundar").

Such targeted error analysis is the basis for deciding whether to fine-tune, replace
the model, or add rule-based fixes in any real-world NER pipeline.
